# Practice Session PS06: Interactions-based recommendation

We will build and apply an item-based collaborative filtering recommenders for movies. 

This practice is longer than the previous ones, and as a side effect will make you an expert in movies from the early 2000s 🍿

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Your name here</font>

E-mail: <font color="#2680EB">Your e-mail here</font>

AI used: <font color="#2680EB">Maker and model, or "none".</font>

Date: <font color="#2680EB">The current date here</font>

# 1. The Movies dataset

This time we will use a different dataset: the [32M version of Movielens](https://www.kaggle.com/datasets/justsahil/movielens-32m), which was released in 2024, but contains user ratings on movies up to 2007 only.

To save computation time, we will use a sub-set of this dataset that contains only 10% of the users and all of their ratings, and that is focused on movies after 2000.

**MOVIES** are described in `ml32m-movies-2000s.csv.gz` in the following format: `movieId,title,genres`.

**RATINGS** are contained in `ml32m-ratings-2000s.csv.gz` in the following format: `userId,movieId,rating`

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

# 1.1. Load the input files

In [ ]:
# LEAVE THIS CODE AS-IS
# But feel free to add imports in an extra cell if needed

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from math import *
import random
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# LEAVE THIS CODE AS-IS

FILENAME_MOVIES = "ml32m-movies-2000s.csv.gz"
FILENAME_RATINGS = "ml32m-ratings-2000s.csv.gz"

In [ ]:
# LEAVE AS-IS

# Load movies
movies = pd.read_csv(FILENAME_MOVIES, 
                    compression='gzip',
                    sep=',', 
                    engine='python', 
                    encoding='utf-8',
                    names=['movie_id', 'title', 'genres'])

# Remove header row from this file
movies.drop(index=0, inplace=True)

# Fix encoding issues in title
# The instructor used AI to fix this encoding issue (Gemini 3.6 Flash)
movies['title'] = movies['title'].str.encode('latin1').str.decode('utf-8')

# Make sure the movie id is numeric
movies["movie_id"] = pd.to_numeric(movies["movie_id"])
print(f"Loaded {movies.shape[0]} movies")
display(movies.sample(10))

In [ ]:
# LEAVE AS-IS

# Load ratings
ratings_raw = pd.read_csv(FILENAME_RATINGS, 
                    sep=',', 
                    compression='gzip',
                    encoding='utf-8',
                    engine='python',
                    names=['user_id', 'movie_id', 'rating'])
display(ratings_raw.head(5))

In [ ]:
# LEAVE AS-IS

# Print some stats on the ratings data
n_users = ratings_raw.user_id.unique().shape[0]
n_movies = ratings_raw.movie_id.unique().shape[0]
print(f"Users who have rated a movie : {n_users}")
print(f"Movies that have been rated  : {n_movies} of {movies.shape[0]}")

# 1.2. Merge the data into a single dataframe <font color="#2680EB">[5%]</font>

Join the data into a single dataframe that should contain columns: user_id, movie_id, rating, title, genres.

You can use Pandas' [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) function to perform an *inner* join.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:

# YOUR CODE HERE

ratings.head(5)

<font size="+1" color="#2680EB">**Copy your implementation of find_movies from the previous practice.** Note that in this dataset the id of a movie is named *movie_id*.</font>

In [ ]:
def find_movies(keyword, movies):
    # YOUR CODE HERE
    
find_movies("Rouge", movies)

The following functions, which you can leave as-is, prints the title of a movie given its movie_id, and gets the movie_id of a movie given its title.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

def get_title(movie_id, movies):
    return movies[movies['movie_id'] == movie_id].title.iloc[0]

def get_genres(movie_id, movies):
    return movies[movies['movie_id'] == movie_id].genres.iloc[0]

def get_movie_id_by_title(title, movies):
    match = movies.loc[movies['title'] == title, 'movie_id']
    return match.iloc[0] if not match.empty else None

# Should find the movie "Idiocracy (2006)"
print(f"Title of movie 47997  : {get_title(47997, movies)}")
print(f"Genres of movie 47997 : {get_genres(47997, movies)}")
print(f"ID of movie 'Idiocracy (2006)': {get_movie_id_by_title("Idiocracy (2006)", movies)}")

# 2. Item-based Collaborative Filtering

The two main types of interactions-based recommender system, also known as *collaborative filtering* algorithms are:

1. **User-based Collaborative Filtering**: To recommend items for user A, we first look at other users B1, B2, ..., Bk with a similar behavior to A, and aggregate their preferences. For instance, if all Bi like a movie that A has not watched, it would be a good candidate to be recommended. 


2. **Item-based Collaborative Filtering**: To recommend items for user A, we first look at all the items I1, I2, ..., Ik that the user A has consumed, and find items that elicit similar ratings from other users. For instance, an item that is rated positively by the same users that rate positively the Ii items, and negatively by the same users that rate negatively the Ii items, would be a good candidate to be recommended.

In both cases, a similarity matrix needs to be built. For user-based, the **user-similarity matrix** will consist of some **distance metrics** that measure the similarity between any two pairs of users. For item-based, the **matrix** will measure the similarity between any two pairs of items.

As we already know, there are several metrics strategy for measure the "similarity" of two items. Some of the most used metrics are Jaccard, Cosine and Pearson. Meanwhile, Jaccard similarity is based on the number of users which have rated item A and B divided by the number of users who have rated either A or B (very useful for those use cases where there is not a numeric rating but just a boolean value like a product being bought), in Pearson and Cosine similarities we measure the similarity between two vectors.

For the purpose of this assignment, we will use **Pearson Similarity** and we will implement a **Item-based Collaborative filtering**.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 2.1. Data pre-processing <font color="#2680EB">[15%]</font>

Now, using the `ratings` dataframe, create a new dataframe named `ratings_summary` containing the following columns:

* movie_id
* title
* ratings_mean (average rating)
* ratings_median (median rating)
* ratings_count (number of people who have rated this movie)

You can use the [agg](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html) function to compute efficiently aggregates ("mean", "median" and "count").
    
<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Generate ratings_summary.**</font>

In [ ]:
ratings_grouped = ratings.groupby(['movie_id', 'title', 'genres'])

# YOUR CODE HERE

display(ratings_summary.sample(10))

Print the 10 movies having the highest and lowest median rating, among the ones having at least 1000 ratings. Given that many movies have the same median rating, break ties using mean rating.

* To select from dataframe A those having column C larger or equal to N, you can do `A[A.C >= N]`.
* To sort dataframe A by decreasing values of columns C (and then D), and select the top N values, you can do `A.sort_values(by=['C', 'D'], ascending=[False, False]).head(N)`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Print the best and worst movies.** Remember to consider only movies having at least 1000 ratings.</font>

In [ ]:
def get_best_movies(min_ratings, max_movies=10):
    # YOUR CODE HERE

def get_worst_movies(min_ratings, max_movies=10):
    # YOUR CODE HERE

min_ratings = 1000
print(f"Best movies by rating (at least {min_ratings} ratings)")
display(get_best_movies(min_ratings))
print()
print(f"Worst movies by rating (at least {min_ratings} ratings)")
display(get_worst_movies(min_ratings))

Notice that this list depends substantially on the minimum number of ratings. If, for instance, we set this to just 3 ratings, the list is quite different.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

min_ratings = 3
print(f"Best movies by rating (at least {min_ratings} ratings)")
display(get_best_movies(min_ratings))
print()
print(f"Worst movies by rating (at least {min_ratings} ratings)")
display(get_worst_movies(min_ratings))

<font size="+1" color="#2680EB">**Compare the lists with min_ratings=1000 against the lists with min_ratings=3.** Answer these questions.</font>

**Answer:**

What would you say about the lists above? 

* (YOUR ANSWER HERE.)

Why do you think this happens?

* (YOUR ANSWER HERE.)


## 2.2. Create the utility matrix <font color="#2680EB">[10%]</font>

Next we will create an **utility matrix**, which we will call `user_movie`, where:

* *columns* are movies,
* *rows* are users,
* each movie-user *cell* contains the score ("utility") of that user for that movie.

We will use the [pivot_table](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html) function of Pandas, which receives a dataframe plus the variable that will make the rows, the variable that will make the columns, and the variable that will make the cells, and transform it into a matrix of the specified rows, columns, and cells.

For instance, if you have a dataframe D containing:

```
U V W
1 a 3.0
1 b 2.0
2 a 1.0
2 c 4.0
```

Calling `D.pivot_table(index='U', columns='V', values='W')` will create the following:

```
V  a   b   c
U
1 3.0 2.0 NaN
2 1.0 NaN 4.0
```

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create the utility matrix.** Use the "pivot_table" function. This may take a few seconds, perhaps up to a minute, to compute.</font>

In [ ]:

# YOUR CODE HERE

user_movie.head(10) 

<font size="+1" color="#2680EB">**Notice that most cells are null.** Answer these questions.</font>

**Answer:**

Why do you think the `user_movie` matrix has so many null values?

* (YOUR ANSWER HERE.)

How do we call this characteristic of user ratings in recommender systems?

* (YOUR ANSWER HERE.)


## 2.3. Explore some correlations in the user-movie matrix <font color="#2680EB">[20%]</font>

Now let us explore whether correlations in this matrix make sense.

1. Consider these three animated movies:
  * *Finding Nemo (2003)* -- we name this id_pivot
  * *Animatrix, The (2003)* -- we name this id_m1
  * *Ratatouille (2007)* -- we name this id_m2
2. Obtain the ratings for each of these movies: `user_movie[movie_id].dropna()`. You will obtain a *series* of ratings for that movie.
3. Consolidate these three series into a single dataframe: `ratings_anim_sample = pd.concat([s1, s2, s3], axis=1)`
4. Drop from `ratings_anim_sample` all rows containing a *NaN*. This will keep only the users that have rated all the 3 movies.
5. Display the first 10 rows from this table.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

id_pivot = get_movie_id_by_title("Finding Nemo (2003)", movies)
id_m1 = get_movie_id_by_title("Animatrix, The (2003)", movies)
id_m2 = get_movie_id_by_title("Ratatouille (2007)", movies)

for movie_id in [id_pivot, id_m1, id_m2]:
    print(f"id: {movie_id:7d}, title: {get_title(movie_id, movies)}")

<font size="+1" color="#2680EB">**Create the "ratings_anim_sample" dataframe.**</font>

In [ ]:
# YOUR CODE HERE

display(ratings_anim_sample)

<font size="+1" color="#2680EB">**Answer the following questions.**</font>

**Answer:**

How many users have rated all the 3 movies?

* (YOUR ANSWER HERE.)

Do you think this is a usual or unusual number, in comparison with a random sample of 3 movies? Why?

* (YOUR ANSWER HERE.)

To compute the **similarity** between the movies, we will use the **correlation between their ratings**. We will assume that two movies are similar if ratings given to them by pairs of users are correlated (i.e., when one user gives a high score, another user also gives a high score, and when one user gives a low score, another user also gives a low score).

To compute Pearson correlation, we use the [corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.corr.html) method. The correlation between two columns `a`, `b` in dataframe `df` is `df[a].corr(df[b])`.

Compute the correlations between all pairs of columns of the `ratings_anim_sample` table.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Compute pair-wise correlations between the ratings of the three movies.**</font>

In [ ]:

for pair in [(id_pivot, id_m1), (id_pivot, id_m2), (id_m1, id_m2)]:
    # YOUR CODE HERE
    print(f"Correlation between ratings of '{get_title(pair[0], movies)}' and '{get_title(pair[1], movies)}': {correlation:.4f}")

<font size="+1" color="#2680EB">**Comment on the correlations found.** Why do you think one of them is higher than the others?</font>

**Answer:**

Why one of these correlations is higher than the others?

* (YOUR ANSWER HERE.)

Now let us take the first movie selected above, the one with movie_id `id_pivot`.

Select the column corresponding to this movie in `user_movies` and compute its correlation with all other columns in `user_movies`. This can be done with  [corrwith](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corrwith.html).

To extract the ratings for movie with id `id_movie` into a dataframe containing a single column `rating`, you can use:

```
df = pd.DataFrame(user_movie[id_movie].dropna()).rename(columns={id_movie: "rating"})
```

To compute the correlation between two single-column dataframes `df1` and `df2`, you can use:

```
corr = df1.corrwith(df2).iloc[0]
```

Store the result in a new dataframe named `correlation_with_pivot` containing two columns: `movie_id` and `corr_with_pivot`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create the "correlation_with_pivot" dataframe.** It should contain columns "movie_id" and "corr_with_pivot". </font>

In [ ]:

# Obtain the ratings of the pivot
ratings_pivot = pd.DataFrame(user_movie[id_pivot].dropna()).rename(columns={id_pivot: "rating"})

correlation_with_pivot_arr = []

for id_movie in user_movie:
    # Obtain the ratings of the other movie
    # Append to correlation_with_pivot_arr an array [id_movie, correlation]
    #  where correlation is the computed correlation between ratings_pivot and the ratings of the other movie
    
    # YOUR CODE HERE

# Turn this into a dataframe
correlation_with_pivot = pd.DataFrame(correlation_with_pivot_arr, columns=["movie_id", "corr_with_pivot"])

# Drop null values
correlation_with_pivot = correlation_with_pivot.dropna(subset=['corr_with_pivot'])

# Display
display(correlation_with_pivot)

Next, create a dataframe `correlation_with_pivot_info` by joining (merging) `correlation_with_pivot` and `ratings_summary` on the column `movie_id`, so that we can add information about title and ratings. This resulting dataframe should have the following columns:

* movie_id
* corr_with_pivot - the correlation between movies movie_id and id_pivot
* title
* ratings_mean
* ratings_count

Keep only rows in which *ratings_count* > 1000, i.e., popular movies. Remember that to filter a dataframe `df` and keep only rows having column `c` larger than `x`, use `df[df[c] > x]`.

We will print the top 5 and bottom 5 rows with the largest and smallest correlation.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create the "correlation_with_pivot_info" dataframe.** Do it by performing an inner join between "correlation_with_pivot" and "ratings_summary".</font>

In [ ]:
# YOUR CODE HERE

print("Movies with high correlation")
display(correlation_with_pivot_info.sort_values('corr_with_pivot', ascending=False).head(5))

print("Movies with low correlation")
display(correlation_with_pivot_info.sort_values('corr_with_pivot').head(5))

<font size="+1" color="#2680EB">**Answer the questions below about these lists.**</font>

**Answer:**

How would you characterize the movies with high correlation?

* (YOUR ANSWER HERE.)

One of the movies with low correlation seems to have a similar theme to the pivot movie, why do you think it has low correlation?

* (YOUR ANSWER HERE.)

# 3. Select two interesting users

We will select two interesting users, for whom we can say that they have different taste in terms of movies.

We will need some auxiliary functions that are provided below.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# Gets the rating a user_id has given to a movie_id
def get_rating(user_movie, user_id, movie_id):
    # The instructor used AI to detect keys that could not be found and return a nan (Gemini 3.6 Flash)
    try:
        val = user_movie[movie_id][user_id]
        return None if pd.isna(val) else val
    except KeyError:
        return None

# Gets a list of movie_ids rated by a user
def get_rated_movie_ids(user_movie, user_id):
    return list(user_movie.loc[user_id].dropna().sort_values(ascending=False).index)

# Get rated movies as a Dataframe
def get_rated_movies(user_movie, movies, user_id):
    rated_movies = get_rated_movie_ids(user_movie, user_id)
    
    df_rated = pd.DataFrame([
        {
            'movie_id': movie_id,
            'rating': get_rating(user_movie, user_id, movie_id),
            'title': get_title(movie_id, movies),
            'genres': get_genres(movie_id, movies)
        } for movie_id in rated_movies
    ])
    return df_rated


## 3.1. Example selection <font color="#2680EB">[5%]</font>

This part shows you an example of how to select two interesting users.

Our first user, `user_id_harry` will be someone who has given the first three *Harry Potter* films a rating higher than 4.5:

* *Harry Potter and the Chamber of Secrets (2002)*
* *Harry Potter and the Prisoner of Azkaban (2004)*
* *Harry Potter and the Goblet of Fire (2005)*

Our second user, `user_id_dark` will be someone who has given the following three dark films a rating higher than 4.5:

* *Requiem for a Dream (2000)*
* *American Psycho (2000)*
* *Donnie Darko (2001)*
* and that has NOT rated the first Harry Potter movie, i.e., having `user_movie[hp_movie_1].isnull()`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# User who likes Harry Potter movies

hp_movie_1 = get_movie_id_by_title("Harry Potter and the Chamber of Secrets (2002)", movies)
hp_movie_2 = get_movie_id_by_title("Harry Potter and the Prisoner of Azkaban (2004)", movies)
hp_movie_3 = get_movie_id_by_title("Harry Potter and the Goblet of Fire (2005)", movies)

list_hp = user_movie[(user_movie[hp_movie_1] > 4.5)
                        & (user_movie[hp_movie_2] > 4.5)
                        & (user_movie[hp_movie_3] > 4.5)].index.values.tolist()
user_id_hp = random.choice(list_hp)

print("There are %d users who really like Harry Potter movies" % len(list_hp))
print("Selected user: user_id=%d" % user_id_hp)

# User who likes dark movies

dark_movie_1 = get_movie_id_by_title("Requiem for a Dream (2000)", movies)
dark_movie_2 = get_movie_id_by_title("American Psycho (2000)", movies)
dark_movie_3 = get_movie_id_by_title("Donnie Darko (2001)", movies)


list_dark = user_movie[(user_movie[dark_movie_1] > 4.5)
                        & (user_movie[dark_movie_2] > 4.5)
                        & (user_movie[dark_movie_3] > 4.5)
                        & (user_movie[hp_movie_1].isnull())].index.values.tolist()
user_id_dark = random.choice(list_dark)

print("There are %d users who like dark movies and have not rated Harry Potter" % len(list_dark))
print("Selected user: user_id=%d" % user_id_dark)


The next code, that you should leave as-is, checks that these users are correctly selected.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

# We check that the selected Harry Potter fan is correct
assert get_rating(user_movie, user_id_hp, hp_movie_1) > 4.5
assert get_rating(user_movie, user_id_hp, hp_movie_2) > 4.5
assert get_rating(user_movie, user_id_hp, hp_movie_3) > 4.5

# We check that the selected dark movies fan is correct
assert get_rating(user_movie, user_id_dark, dark_movie_1) > 4.5
assert get_rating(user_movie, user_id_dark, dark_movie_2) > 4.5
assert get_rating(user_movie, user_id_dark, dark_movie_3) > 4.5
assert pd.isna(get_rating(user_movie, user_id_dark, hp_movie_1))

Let us look closely at the interests of these selected users. Note that they have rated many movies, so they tastes may be more complex than the reasons why we selected them.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS-IS

print("Some of the movies rated highly by the user who like Harry Potter movies")
display(get_rated_movies(user_movie, movies, user_id_hp).sort_values(by='rating', ascending=False).head(20))

print("Some of the movies rated highly by the user who like dark movies")
display(get_rated_movies(user_movie, movies, user_id_dark).sort_values(by='rating', ascending=False).head(20))


<font size="+1" color="#2680EB">**Describe how you would characterize the tastes of these two users.**</font>

**Answer:**

How would you characterize the tastes of the first user?

* (YOUR ANSWER HERE.)

How would you characterize the tastes of the second user?

* (YOUR ANSWER HERE.)

## 3.2. Your selection <font color="#2680EB">[10%]</font>

Now, do it yourself. Select two users, `user_id_1` and `user_id_2`, who have different tastes. You can use the previous section as a template for this, but do not need to do it in exactly the same manner.

Basically, pick two sets of around 3 movies. For each set, there should many users who have rated all movies of the set, ideally. Pick an user from them, print the movies that this user like, and try to characterize the users.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement your code to find these users.** Make sure the titles of the movies that you are using to guide this search are visible in your code. You do not need to do this exactly as we did above, but just make sure you select two users that have clearly different tastes.</font>

In [ ]:
# YOUR CODE HERE

<font size="+1" color="#2680EB">**Describe how you would characterize their tastes** based on the ratings they give to different movies.</font>

**Answer:**

How would you characterize the tastes of your first user, having `user_id=user_id_1`?

* (YOUR ANSWER HERE.)

How would you characterize the tastes of your second user, having `user_id=user_id_2`?

* (YOUR ANSWER HERE.)

# 4. Generate recommendations

We are ready now to generate recommendations.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 4.1. Compute item similarities <font color="#2680EB">[5%]</font>

Let us compute all similarities between movies in the utility matrix `user_movie`.

To compute all correlations between columns in a dataframe, use [corr](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html). This function receives a matrix with *r* rows and *c* columns, and returns a square matrix of *c x c* containing all pair-wise correlations between columns.

However, this computation can be unreliable if the columns do not have many elements in common. Fortunately, we can use parameter `min_periods` that indicates the minimum number of elements in common that two columns must have to compute the correlation.

**⚠️ This process may take a few minutes.**

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Compute all Pearson correlations between columns in "user_movie".** Restrict this to columns having at least 100 elements in common.</font>

In [ ]:
# YOUR CODE HERE

item_similarity.head(5)

## 4.2. Compute the relevance of movies not yet rated by a user <font color="#2680EB">[20%]</font>

We will consider that the importance of an **unseen** movie (i.e., a movie that a user has not yet rated) will be equal to the **sum of the similarities** between that new movie and all the movies the user has already rated.

To further improve this, we will compute a weighted sum, in which the weight will be the rating given to the movie.

For instance, suppose a user has rated movies as follows:

```
movie_id rating
1        2.0
2        3.0
3        NaN
4        NaN
```

And that movie similarities are given by the following square matrix (values with a "." do not matter in this example).

```
movie_id   1   2   3   4
1         ...............
2         ...............
3         0.1 0.2 NaN ...
4         0.9 0.8 ... NaN
```

The importance of movie 3 to this user will be:

```
2.0 * 0.1 + 3.0 * 0.2 = 0.8
```

While the importance of movie 4 to this user will be:

```
2.0 * 0.9 + 3.0 + 0.8 = 5.6
```

As we can see, we are favoring movies that are highly similar to many movies that the user has rated high.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement the movies relevance function.** It should return a dataframe with columns "movie_id", "relevance".</font>

In [ ]:
def get_movies_relevance(user_id, user_movie, item_similarity):
    
    # Create an empty series
    movies_relevance = pd.Series(dtype='float64')
    
    # Iterate through the movies the user has watched
    rated_movies = get_rated_movie_ids(user_movie, user_id)
    
    for rated_movie in rated_movies:
        
        # 1. Obtain the rating given to the rated_movie
        # 2. Obtain the similarities to all other movies
        # 3. Multiply these into weighted_similarities
        
        # YOUR CODE HERE
        
        # Append these terms to movies_relevance
        movies_relevance = pd.concat([movies_relevance, weighted_similarities])
    
    # Compute the sum of weights for each movie,
    # remember that the series is indexed by movie_id
    movies_relevance = movies_relevance.groupby(movies_relevance.index).sum()
    
    # Convert to a dataframe
    # The instructor used AI to convert the series to a dataframe and reindex (Gemini 3.6 Flash)
    movies_relevance_df = pd.DataFrame(movies_relevance, columns=['relevance']).reset_index(names='movie_id')
    
    # Remove movies that the user has rated
    rows_to_remove = movies_relevance_df['movie_id'].isin(rated_movies)
    movies_relevance_df = movies_relevance_df[~rows_to_remove]
    
    return movies_relevance_df

## 4.3. Generate recommendations for your two users <font color="#2680EB">[10%]</font>

Now we use `get_movies_relevance` to obtain recommendations for the selected users (`user_id_1` and `user_id_2`).

The output of this function contains only `movie_id` and `relevance`, so we merge it with the `movies` dataframe on the `movie_id` attribute, and sort by descending relevance.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [ ]:
# LEAVE AS=IS

print(f"User 1 (user_id={user_id_1})")
rel = get_movies_relevance(user_id_1, user_movie, item_similarity)
rel = rel.merge(movies, on='movie_id')
display(rel.sort_values('relevance', ascending=False).head(10))

print(f"User 2 (user_id={user_id_1})")
rel = get_movies_relevance(user_id_2, user_movie, item_similarity)
rel = rel.merge(movies, on='movie_id')
display(rel.sort_values('relevance', ascending=False).head(10))

<font size="+1" color="#2680EB">**Comment on these recommendations.** Indicate if they are relevant in general, if there is any recommendation that looks particularly correct or particularly wrong for a user, and any other comment you may have.</font>

**Answer:**

What are your comments on the recommendations for your first user, having `user_id=user_id_1`? 

* (YOUR ANSWER HERE.)

What are your comments on the recommendations for your first user, having `user_id=user_id_2`?

* (YOUR ANSWER HERE.)

# DELIVER (individually)

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver this notebook, renamed as `PSxx_Unnnnnn_YourName.ipynb` where:
* *PSxx* is the practice number.
* *Unnnnnn* is your UPF id number.
* *YourName* is your name.

## Extra points available

For more learning and extra points, use the [surprise](http://surpriselib.com/) library to generate recommendations for the same two users. You may want to start by looking at [loading a custom dataset from a dataframe](https://surprise.readthedocs.io/en/stable/getting_started.html#use-a-custom-dataset), which converts a utility matrix into the dataset structure that surprise uses. Generate recommendations using any algorithm that uses the utility matrix, and comment on these recommendations for your selected users.

**Note:** if you go for the extra points, add ``<font size="+2" color="#2680EB">Additional results: surprise library</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the AI instructions stated above.</font>